# 7 skyrius: Pokalbių programų kūrimas
## Microsoft Foundry modelių API greitasis pradžiamokslis

Šis užrašų knygelė yra pritaikyta iš [Azure OpenAI pavyzdžių saugyklos](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), kuri apima užrašų knygeles, prieinamas per [Azure OpenAI](aoai-assignment.ipynb) paslaugas.

> **Pastaba:** GitHub modeliai nutraukiami 2026 m. liepos pabaigoje. Ši užrašų knygelė dabar orientuota į [Microsoft Foundry modelius](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), kurie siūlo tą patį nemokamai išbandyti skirtą modelių katalogą ir Azure AI Inference SDK patirtį.


# Apžvalga  
"Dideli kalbos modeliai yra funkcijos, kurios žemėlapiuoja tekstą į tekstą. Turint įvesties teksto eilutę, didelis kalbos modelis bando nuspėti, koks tekstas ateis po jos"(1). Ši „greito pradžios“ užrašų knygelė supažindins vartotojus su aukšto lygio LLM sąvokomis, pagrindiniais AML paketų reikalavimais pradėti, minkštu įvadu į užklausų kūrimą ir keliais trumpais įvairių naudojimo atvejų pavyzdžiais. 


## Turinys  

[Apžvalga](#overview)  
[Kaip naudotis OpenAI paslauga](#how-to-use-openai-service)  
[1. Sukurkite savo OpenAI paslaugą](#1.-creating-your-openai-service)  
[2. Įdiegimas](#2.-installation)    
[3. Kredencialai](#3.-credentials)  

[Naudojimo atvejai](#use-cases)    
[1. Teksto santraukų sudarymas](#1.-summarize-text)  
[2. Teksto klasifikavimas](#2.-classify-text)  
[3. Naujų produktų pavadinimų generavimas](#3.-generate-new-product-names)  
[4. Klasifikatoriaus patikslinimas](#4.fine-tune-a-classifier)  

[Nuorodos](#references)


### Sukurkite savo pirmą užklausą  
Ši trumpa užduotis suteiks pagrindinį įvadą, kaip pateikti užklausas modeliui „Microsoft Foundry Models“ paprastam užduoties „santrauka“ atlikimui.


**Veiksmai**:  
1. Įdiekite `azure-ai-inference` biblioteką savo „python“ aplinkoje, jei to dar nepadarėte.  
2. Įkelkite standartines pagalbines bibliotekas ir nustatykite savo „Microsoft Foundry Models“ kredencialus.  
3. Pasirinkite modelį savo užduočiai  
4. Sukurkite paprastą užklausą modeliui  
5. Pateikite savo užklausą modeliui per API!


### 1. Įdiegti `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Importuokite pagalbines bibliotekas ir sukurkite kredencialus


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Tinkamo modelio radimas  
Tokios modeliai kaip GPT-4o ir GPT-4o mini gali suprasti ir generuoti natūralų kalbą, ir yra prieinami Microsoft Foundry Models kataloge kartu su Meta, Mistral, Cohere ir Microsoft modeliais.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Užklausos kūrimas  

"Didžiųjų kalbos modelių magija yra ta, kad juos mokant sumažinti šią numatymo klaidą didžiuliais tekstų kiekiais, modeliai galiausiai išmoksta koncepcijų, naudingų šiems numatymams. Pavyzdžiui, jie išmoksta tokių koncepcijų kaip"(1):

* kaip taisyklingai rašyti
* kaip veikia gramatika
* kaip perfrazuoti
* kaip atsakyti į klausimus
* kaip palaikyti pokalbį
* kaip rašyti daugeliu kalbų
* kaip programuoti
* ir t.t.

#### Kaip valdyti didelį kalbos modelį  
"Iš visų įvesties duomenų dideliam kalbos modeliui, daugiausia įtakos turi tekstinė užklausa(1).

Didelius kalbos modelius galima paskatinti generuoti atsakymus keliais būdais:

Instrukcija: Pasakykite modeliui, ko norite
Užbaigimas: Paskatinkite modelį užbaigti tai, ko norite pradžioje
Demonstruoti: Parodykite modeliui, ko norite, naudodami:
Keletą pavyzdžių užklausoje
Daug šimtų ar tūkstančių pavyzdžių, esančių tobulinimo mokymo duomenų rinkinyje"



#### Yra trys pagrindinės gairės kuriant užklausas:

**Rodykite ir sakykite**. Aiškiai parodykite, ko norite, arba per instrukcijas, pavyzdžius, arba jų derinį. Jei norite, kad modelis surikiuotų sąrašą abėcėlės tvarka arba klasifikuotų pastraipą pagal nuotaiką, parodykite, kad tai yra jūsų pageidavimas.

**Pateikite kokybiškus duomenis**. Jei bandote sukurti klasifikatorių ar priversti modelį laikytis tam tikro šablono, įsitikinkite, kad pavyzdžių pakanka. Būtinai peržiūrėkite savo pavyzdžius – modelis dažniausiai yra pakankamai išmanus, kad matytų elementarias rašybos klaidas ir pateiktų atsakymą, bet taip pat gali manyti, kad tai yra sąmoningas veiksmas, o tai gali paveikti atsakymą.

**Tikrinkite savo nustatymus.** Temperatūros ir top_p nustatymai kontroliuoja, kiek modelis yra deterministiškas generuodamas atsakymą. Jei prašote atsakymo, kuriam yra tik vienas teisingas sprendimas, norėsite šiuos parametrus sumažinti. Jei norite įvairesnių atsakymų, tuomet galite pakelti šiuos nustatymus. Didžiausia žmonių daroma klaida su šiais nustatymais yra manymas, kad jie valdo „sumanumą“ ar „kūrybiškumą“.


Šaltinis: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Pateikite!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Pakartokite tą pačią užklausą, kaip palygina rezultatai?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Santrauka teksto  
#### Iššūkis  
Santraukuokite tekstą pridėdami 'tl;dr:' pabaigoje prie teksto pastraipos. Atkreipkite dėmesį, kaip modelis supranta, kaip atlikti daugybę užduočių be papildomų nurodymų. Galite eksperimentuoti su labiau apibūdinančiais raginimais nei tl;dr, kad modifikuotumėte modelio elgesį ir suasmenintumėte gaunamą santrauką(3).  

Naujausi darbai parodė žymų pažangą daugelyje NLP užduočių ir testų, iš anksto apmokant didelį teksto korpusą, o po to tiksliai derinant konkrečiai užduočiai. Nors architektūra paprastai būna užduočių nesiimanti, šis metodas vis tiek reikalauja užduočių specifinių derinimo duomenų rinkinių, turinčių tūkstančius ar dešimtis tūkstančių pavyzdžių. Tuo tarpu žmonės paprastai sugeba atlikti naują kalbos užduotį tik iš kelių pavyzdžių arba paprastų nurodymų – ką dabartinės NLP sistemos vis dar dažniausiai sunkiai sugeba. Čia mes parodome, kad kalbos modelių mastelio didinimas žymiai pagerina užduočių nesiimančius, kelių pavyzdžių atliekamus veiksmus, kartais net pasiekdamas konkurencingumą su ankstesniais pažangiausiais tikslinio derinimo metodais. 



Santrauka


# Pratimai keliems naudojimo atvejams  
1. Teksto santrauka  
2. Teksto klasifikavimas  
3. Naujų produktų pavadinimų kūrimas


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Teksto klasifikavimas  
#### Užduotis  
Priskirkite elementus kategorijoms, pateiktoms sprendimo metu. Žemiau pateiktame pavyzdyje pateikiame tiek kategorijas, tiek tekstą klasifikavimui užklausoje(*playground_reference). 

Kliento užklausa: Sveiki, neseniai mano nešiojamojo kompiuterio klaviatūros vienas mygtukas sulūžo ir man reikės pakaitalo:

Priskirta kategorija:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Sugeneruokite naujus produktų pavadinimus
#### Iššūkis
Sukurkite produktų pavadinimus pagal pateiktus žodžius. Čia užklausoje įtraukiame informaciją apie produktą, kuriam norime sugeneruoti pavadinimus. Taip pat pateikiame panašų pavyzdį, kad parodytume norimą gavimo modelį. Temperatūros reikšmę nustatėme aukštą, kad padidintume atsitiktinumą ir gautume inovatyvesnius atsakymus.

Produkto aprašymas: namų pieno kokteilių gamintuvas
Pradiniai žodžiai: greitas, sveikas, kompaktiškas.
Produktų pavadinimai: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Produkto aprašymas: pora batų, tinkančių bet kokio dydžio kojai.
Pradiniai žodžiai: prisitaikantis, tinkantis, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Nuorodos  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Geriausios praktikos GPT modelių tiksliniam pritaikymui tekstų klasifikavimui](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Daugiau pagalbos  
[OpenAI Komercializavimo komanda](AzureOpenAITeam@microsoft.com) 


# Dalyviai
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
